# Medical Image Classification Challenge 2026 — Competition Baseline v1

**Task:** binary chest X-ray classification: `Normal` vs `Pneumonia`.

**Rules-first design:** Use only data provided by organizers; no test labels; no external medical images; do **not** install packages or download weights. Train entirely within Kaggle. This version uses **randomly initialized torchvision models** (`weights=None`), not unauthorized pretrained weights.

**Evaluation:** `P = 0.30*Accuracy + 0.25*Sensitivity + 0.20*Specificity + 0.25*F1`, with **Pneumonia as positive class (1)**. A single test CSV is generated automatically from a reloaded `.pkl` model, never edited by hand.

### Before running
1. Attach the organizer-provided dataset to Kaggle Notebook and enable GPU if available.
2. Set `DATA_ROOT` in the config cell and ensure a standard structure, e.g. `train/Normal/`, `train/Pneumonia/`, `val/Normal/`, `val/Pneumonia/`, `test/*.jpg`. If different, adjust the loader deliberately and document it.
3. Run **all cells from top to bottom**. Never examine unseen test labels or select models using hidden test feedback.
4. Notebook is a **starting point**, not a measured winning model. Keep your own experiment log and ensure every team member can explain it.

## 1. Environment, configuration and reproducibility

In [ ]:
import os
import sys
import time
import json
import pickle
import zipfile  # Python standard library; used only for packaging submission

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

# ====== CHANGE THIS PATH TO THE ORGANIZER-PROVIDED DATASET ======
DATA_ROOT = '/kaggle/input/REPLACE_WITH_ORGANIZER_DATASET'
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else './medical_outputs'
TEAM_ID = 'team01'  # Replace with the exact ID required by the organizers

# ====== FIRST EXPERIMENT ======
ARCH = 'resnet18'      # Supported: 'resnet18', 'efficientnet_b0'; NO downloaded weights
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
PATIENCE = 7
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
SEED = 2026
NUM_WORKERS = 0     # Deterministic/simple Kaggle baseline

# Note: changing experiments is allowed, but record every change and validation score.
CLASS_NAMES = ['Normal', 'Pneumonia']
LABEL_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Torchvision model:', ARCH)
print('Device:', DEVICE)
print('Dataset directory:', DATA_ROOT)

## 2. Discover files and audit dataset layout
Labels must come **only** from the organizer-provided train/validation folders. Test labels are never needed or inferred from file names.

In [ ]:
VALID_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

def find_split_directory(root, aliases):
    if not os.path.isdir(root):
        raise FileNotFoundError(
            'DATA_ROOT does not exist. Attach the organizer dataset in Kaggle and update DATA_ROOT: ' + root
        )
    children = {x.lower(): x for x in os.listdir(root) if os.path.isdir(os.path.join(root, x))}
    for alias in aliases:
        if alias.lower() in children:
            return os.path.join(root, children[alias.lower()])
    raise FileNotFoundError(f'Cannot find split {aliases} in {root}. Found directories: {list(children.values())}')

def get_class_dir(split_root, class_name):
    children = {x.lower(): x for x in os.listdir(split_root) if os.path.isdir(os.path.join(split_root, x))}
    if class_name.lower() not in children:
        raise FileNotFoundError(f'Missing class {class_name} in {split_root}. Found {list(children.values())}')
    return os.path.join(split_root, children[class_name.lower()])

def image_paths(root):
    paths = []
    for directory, subdirs, names in os.walk(root):
        subdirs.sort()
        for name in sorted(names):
            if name.lower().endswith(VALID_EXTENSIONS):
                paths.append(os.path.join(directory, name))
    return sorted(paths)

def get_labeled_samples(split_root):
    result = []
    for name in CLASS_NAMES:
        for path in image_paths(get_class_dir(split_root, name)):
            result.append((path, LABEL_MAP[name]))
    return result

train_root = find_split_directory(DATA_ROOT, ['train', 'training'])
val_root = find_split_directory(DATA_ROOT, ['val', 'validation', 'valid'])
test_root = find_split_directory(DATA_ROOT, ['test', 'testing'])

train_samples = get_labeled_samples(train_root)
val_samples = get_labeled_samples(val_root)
test_paths = image_paths(test_root)

if not train_samples or not val_samples or not test_paths:
    raise ValueError('Train, validation, and test must all contain images.')
if len(set(p for p, _ in train_samples)) != len(train_samples):
    raise ValueError('Duplicate train image paths.')
if len(set(os.path.basename(p) for p in test_paths)) != len(test_paths):
    raise ValueError('Duplicate test basenames. Image_id must be unique; contact organizers.')

train_counts = np.bincount([label for _, label in train_samples], minlength=2)
val_counts = np.bincount([label for _, label in val_samples], minlength=2)
if (train_counts == 0).any() or (val_counts == 0).any():
    raise ValueError('Both Normal and Pneumonia must be present in train and validation.')

print('Train:', len(train_samples), dict(zip(CLASS_NAMES, train_counts.tolist())))
print('Validation:', len(val_samples), dict(zip(CLASS_NAMES, val_counts.tolist())))
print('Test (unlabeled):', len(test_paths))
print('Train class balance:', np.round(train_counts / train_counts.sum(), 3))

# This checks only exact shared paths, not duplicate content or patient overlap.
assert not (set(p for p, _ in train_samples) & set(p for p, _ in val_samples))
print('IMPORTANT: Also check for patient-level overlap and duplicated images if metadata allows.')

## 3. Preview only organizer-provided training images
Check orientation, black borders, annotation marks, image quality, and abnormal label distributions; do not manually modify files.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for row, label in enumerate([0, 1]):
    examples = [p for p, y in train_samples if y == label][:3]
    for col, ax in enumerate(axes[row]):
        ax.axis('off')
        if col < len(examples):
            with Image.open(examples[col]) as im:
                ax.imshow(im.convert('L'), cmap='gray')
            ax.set_title(CLASS_NAMES[label], fontsize=10)
plt.suptitle('Training set: sample inspection (not test)')
plt.tight_layout()
plt.show()

## 4. Preprocessing and augmentation
We convert X-rays to 3-channel grayscale RGB for torchvision architectures (not adding any new image information). Mild rotation, tight crop, and slight brightness/contrast jitter apply **only to training**; validation and test receive identical deterministic transforms. Horizontal flip is omitted due to anatomical laterality concerns.

In [ ]:
# Fixed normalization makes training/evaluation transforms consistent.
MEAN, STD = [0.5] * 3, [0.5] * 3
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 24, IMAGE_SIZE + 24)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.90, 1.0), ratio=(0.98, 1.02)),
    transforms.RandomRotation(degrees=7),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class ChestXrayDataset(Dataset):
    def __init__(self, samples, transform, labeled=True):
        self.samples = samples
        self.transform = transform
        self.labeled = labeled

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        if self.labeled:
            path, label = self.samples[index]
        else:
            path, label = self.samples[index], -1
        with Image.open(path) as im:
            image = self.transform(im.convert('L').convert('RGB'))
        return image, int(label)

train_dataset = ChestXrayDataset(train_samples, train_transform)
val_dataset = ChestXrayDataset(val_samples, eval_transform)
test_dataset = ChestXrayDataset(test_paths, eval_transform, labeled=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')

batch_images, batch_labels = next(iter(train_loader))
print('Sample batch:', batch_images.shape, 'labels:', batch_labels[:8].tolist())

## 5. Architecture and score implementation
We score **Pneumonia = positive (1)**. A balanced loss compensates for class imbalance without artificially altering the validation/test distribution. If training data is balanced, equal class weights are naturally near 1.

In [ ]:
def build_model(architecture):
    if architecture == 'resnet18':
        model = models.resnet18(weights=None)  # no Internet / pretrained downloads
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif architecture == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=None)  # no Internet / pretrained downloads
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, 2)
    else:
        raise ValueError('Unknown architecture: ' + architecture)
    return model

def calculate_metrics(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=float)
    preds = (probabilities >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
    contest_p = 0.30 * accuracy + 0.25 * sensitivity + 0.20 * specificity + 0.25 * f1
    try:
        auc = roc_auc_score(labels, probabilities)
    except ValueError:
        auc = float('nan')
    return dict(P=float(contest_p), Accuracy=float(accuracy), Sensitivity=float(sensitivity),
                Specificity=float(specificity), F1=float(f1), AUROC=float(auc),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn), threshold=float(threshold))

@torch.no_grad()
def predict_probabilities(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        probs = torch.softmax(logits.float(), dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy().tolist())
        all_labels.extend(labels.numpy().tolist())
    return np.array(all_labels, dtype=np.int64), np.array(all_probs, dtype=float)

model = build_model(ARCH).to(DEVICE)
class_weights = train_counts.sum() / (2.0 * train_counts.astype(float))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print('Trainable parameters:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print('Class weights:', dict(zip(CLASS_NAMES, np.round(class_weights, 3))))

## 6. Train, validate, select the best epoch
Checkpoint selection uses the official weighted Performance score at a **predeclared threshold of 0.50**, not unknown test results. Save a detached CPU copy of the best model to make the final `.pkl` self-contained.

In [ ]:
best_score = -1.0
best_epoch = 0
best_state_dict = None
history = []
non_improving_epochs = 0

for epoch in range(1, EPOCHS + 1):
    started = time.time()
    model.train()
    total_loss = 0.0
    seen = 0
    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * len(images)
        seen += len(images)

    scheduler.step()
    _, val_probs = predict_probabilities(model, val_loader)
    val_labels = np.array([label for _, label in val_samples], dtype=int)
    m = calculate_metrics(val_labels, val_probs, threshold=0.5)
    train_loss = total_loss / max(seen, 1)
    history.append({'epoch': epoch, 'loss': train_loss, **m})
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | P={m['P']:.4f} | "
          f"Se={m['Sensitivity']:.4f} Sp={m['Specificity']:.4f} "
          f"F1={m['F1']:.4f} | {time.time()-started:.0f}s")

    if m['P'] > best_score + 1e-8:
        best_score = m['P']
        best_epoch = epoch
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        non_improving_epochs = 0
    else:
        non_improving_epochs += 1
        if non_improving_epochs >= PATIENCE:
            print('Early stopping triggered.')
            break

if best_state_dict is None:
    raise RuntimeError('Training never produced a valid best checkpoint.')
model.load_state_dict(best_state_dict)
print('Best epoch:', best_epoch, '| best validation P @ threshold 0.50:', round(best_score, 5))

## 7. Threshold selection — validation only
Potentially useful because the contest weighs sensitivity, specificity and F1 separately. A small, predeclared grid minimizes overfitting risk. If either validation class has fewer than 50 samples, retain `0.50` rather than fitting an unstable threshold.

In [ ]:
val_labels, val_probs = predict_probabilities(model, val_loader)
fixed_metrics = calculate_metrics(val_labels, val_probs, threshold=0.5)
threshold_grid = np.round(np.arange(0.30, 0.701, 0.025), 3)
threshold_metrics = [calculate_metrics(val_labels, val_probs, t) for t in threshold_grid]
threshold_table = pd.DataFrame(threshold_metrics)[['threshold', 'P', 'Accuracy', 'Sensitivity', 'Specificity', 'F1']]

if np.min(val_counts) < 50:
    best_threshold = 0.50
    print('Small validation set: retain threshold=0.50 to avoid unstable tuning.')
else:
    ranked = sorted(threshold_metrics, key=lambda r: (-r['P'], abs(r['threshold'] - 0.5)))
    best_threshold = float(ranked[0]['threshold'])

best_val_metrics = calculate_metrics(val_labels, val_probs, best_threshold)
print('Fixed threshold P:', round(fixed_metrics['P'], 5))
print('Chosen threshold:', best_threshold)
print('Chosen threshold metrics:', best_val_metrics)
display(threshold_table.round(4))

history_frame = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history_frame['epoch'], history_frame['loss'])
ax[0].set_title('Training loss')
ax[0].set_xlabel('Epoch')
ax[1].plot(history_frame['epoch'], history_frame['P'])
ax[1].set_title('Validation official Performance P @ 0.50')
ax[1].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

## 8. Confusion matrix and error analysis
Visual examples of false positives/false negatives must remain within the organizer's labeled validation set. Explain that heatmaps and image review do not replace clinical validation.

In [ ]:
predictions = (val_probs >= best_threshold).astype(int)
cm = confusion_matrix(val_labels, predictions, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1], CLASS_NAMES)
ax.set_yticks([0, 1], CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('Ground truth')
ax.set_title('Validation confusion matrix')
for row in range(2):
    for col in range(2):
        ax.text(col, row, str(cm[row, col]), ha='center', va='center', color='black')
plt.tight_layout()
plt.show()

def show_errors(true_class, predicted_class, max_images=4):
    wrong_indices = np.where((val_labels == true_class) & (predictions == predicted_class))[0][:max_images]
    if len(wrong_indices) == 0:
        print(f'No {CLASS_NAMES[true_class]} -> {CLASS_NAMES[predicted_class]} errors.')
        return
    fig, axes = plt.subplots(1, len(wrong_indices), figsize=(3 * len(wrong_indices), 3))
    axes = np.atleast_1d(axes)
    for ax, index in zip(axes, wrong_indices):
        path, _ = val_samples[index]
        with Image.open(path) as image:
            ax.imshow(image.convert('L'), cmap='gray')
        ax.set_title(f'Pneumonia p={val_probs[index]:.3f}')
        ax.axis('off')
    plt.suptitle(f'Validation: True {CLASS_NAMES[true_class]} / Pred {CLASS_NAMES[predicted_class]}')
    plt.tight_layout()
    plt.show()

show_errors(true_class=1, predicted_class=0)  # false negatives: missed pneumonia
show_errors(true_class=0, predicted_class=1)  # false positives: normal predicted pneumonia

## 9. Save the model in `.pkl` and verify a reload
Weights and inference settings must travel together. `pickle` files should **only be loaded from your own trusted submission**. Test output is generated from the reloaded checkpoint to guard against missing model parameters.

In [ ]:
model_path = os.path.join(OUTPUT_DIR, 'model.pkl')
checkpoint = {
    'architecture': ARCH,
    'state_dict': best_state_dict,
    'class_names': CLASS_NAMES,
    'image_size': IMAGE_SIZE,
    'mean': MEAN,
    'std': STD,
    'threshold': float(best_threshold),
    'seed': int(SEED),
    'best_epoch': int(best_epoch),
    'best_validation_metrics': best_val_metrics,
    'torch_version': torch.__version__,
}
with open(model_path, 'wb') as file:
    pickle.dump(checkpoint, file, protocol=pickle.HIGHEST_PROTOCOL)

with open(model_path, 'rb') as file:
    restored = pickle.load(file)
assert restored['class_names'] == CLASS_NAMES
restored_model = build_model(restored['architecture']).to(DEVICE)
restored_model.load_state_dict(restored['state_dict'], strict=True)
restored_model.eval()

# Self-check: the reloaded model must have the same probabilities on validation.
_, restored_val_probs = predict_probabilities(restored_model, val_loader)
np.testing.assert_allclose(val_probs, restored_val_probs, atol=1e-6, rtol=1e-6)
print('Saved and reloaded:', model_path)
print('Reload consistency check: PASS')

## 10. Generate unlabeled test predictions — exactly `Image_id`, `Prediction`
Test labels never enter code. Maintain original base filenames. No manual edits. Use **the restored model**, **the validated transform**, and **the selected validation threshold**.

In [ ]:
test_labels_dummy, test_probabilities = predict_probabilities(restored_model, test_loader)
assert np.all(test_labels_dummy == -1), 'Test dataset must be unlabeled.'
assert len(test_probabilities) == len(test_paths)
assert np.isfinite(test_probabilities).all()

test_class_indices = (test_probabilities >= restored['threshold']).astype(int)
submission = pd.DataFrame({
    'Image_id': [os.path.basename(p) for p in test_paths],
    'Prediction': [CLASS_NAMES[i] for i in test_class_indices],
})
assert submission['Image_id'].is_unique
assert set(submission['Prediction']).issubset({'Normal', 'Pneumonia'})
assert submission.shape == (len(test_paths), 2)
assert submission.columns.tolist() == ['Image_id', 'Prediction']

csv_path = os.path.join(OUTPUT_DIR, f'{TEAM_ID}.csv')
submission.to_csv(csv_path, index=False)

# Round-trip validation ensures the submitted CSV content matches the in-memory predictions.
roundtrip = pd.read_csv(csv_path)
pd.testing.assert_frame_equal(submission, roundtrip)
print('CSV ready:', csv_path, '| image count:', len(submission))
display(submission.head())

## 11. Reproducibility summary and final submission packaging
Organizers require one `.zip` with **at least** `model.pkl`, `<team>.csv`, and the runnable `.ipynb`. Kaggle may not expose the current notebook file on disk. If so, **download it from Kaggle UI** and package these three files without modifying the CSV. The auto-ZIP below only executes when the actual notebook source is present.

In [ ]:
run_summary = {
    'data_root': DATA_ROOT,
    'n_train': len(train_samples),
    'n_validation': len(val_samples),
    'n_test': len(test_paths),
    'architecture': ARCH,
    'weights_downloaded': False,
    'seed': SEED,
    'best_epoch': best_epoch,
    'threshold': best_threshold,
    'validation_metrics': best_val_metrics,
    'submission_csv': os.path.basename(csv_path),
}
summary_path = os.path.join(OUTPUT_DIR, 'experiment_summary.json')
with open(summary_path, 'w', encoding='utf-8') as file:
    json.dump(run_summary, file, indent=2, ensure_ascii=False)
print(json.dumps(run_summary, indent=2, ensure_ascii=False))

# Kaggle kernels generally do not expose the source notebook at this path automatically.
# Set this only if you have explicitly copied/exported the actual .ipynb to OUTPUT_DIR.
NOTEBOOK_SOURCE_PATH = os.path.join(OUTPUT_DIR, 'Medical_Xray_Competition_Starter.ipynb')
submission_zip_path = os.path.join(OUTPUT_DIR, TEAM_ID + '_submission.zip')
if os.path.isfile(NOTEBOOK_SOURCE_PATH):
    with zipfile.ZipFile(submission_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for path in [model_path, csv_path, NOTEBOOK_SOURCE_PATH]:
            z.write(path, arcname=os.path.basename(path))
    with zipfile.ZipFile(submission_zip_path) as z:
        assert len(z.namelist()) == 3
        assert any(x.endswith('.ipynb') for x in z.namelist())
    print('VALID 3-FILE ZIP:', submission_zip_path)
else:
    print('NOTEBOOK SOURCE NOT FOUND ON DISK: ZIP not created.')
    print('Download the executed .ipynb from Kaggle, then package it together with:')
    print('  ', model_path)
    print('  ', csv_path)
    print('Please verify the organizer-required filenames before submission.')

---

## Experiment ladder — do not make all changes at once

| Experiment | Intervention | Reason | Compare with |
|---|---|---|---|
| E00 | ResNet18, from scratch, 224px | Strictly compliant baseline | official P, Se, Sp, F1 |
| E01 | Turn off data augmentation | Estimate augmentation benefit | E00 |
| E02 | EfficientNet-B0, from scratch | Alternative architecture | E00 |
| E03 | Image size 288/320 if GPU allows | Retain more lung detail | E00 |
| E04 | CLAHE on train/val/test | Assess contrast changes fairly | E00 |
| E05 | One permitted pretrained checkpoint **only with written organizer approval** | Test transfer learning | E00 |
| E06 | Ensemble only strong, complementary models | Better generalization, not just validation score | individual models |

**Scientific controls:** Record train/validation size and patient separation, seed, augmentation, architecture, learning rate, best epoch, chosen threshold, confusion matrix and execution time. Choose final changes using validation only; no test labels, no third-party data, no dataset/model exchange across teams. Do not apply aggressive crops that clip lung anatomy. If patient IDs exist, check for patient-level overlap between splits and alert organizers if found.

**Clinical limitation:** This is a competition classifier, not a diagnostic medical device. Avoid presenting validation score as proof of clinical deployment safety.